# Figure 1A

Environment: `python_env` (`envs/python_env.yml`)

Input: `data/CFU_PFU.xlsx`

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42

ROOT = Path("..")
config = yaml.safe_load(open(ROOT / "config.yaml"))
OUT_DIR = Path(config["out_dir"])
FIG_DIR = OUT_DIR / "figures"
SRC_DIR = OUT_DIR / "source_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_excel(ROOT / "data" / "CFU_PFU.xlsx", index_col=0)

## Figure 1A

In [ ]:
groups = {
    "CFU - Control (LB)": [c for c in df.columns if "Control" in c],
    "CFU - Infected (MOI=10)": [c for c in df.columns if "Infected" in c],
    "PFU": [c for c in df.columns if "PFU" in c],
}
colors = {
    "CFU - Control (LB)": "#1f77b4",
    "CFU - Infected (MOI=10)": "#ff7f0e",
    "PFU": "black",
}

plt.figure(figsize=(10, 5))
summary = []

for name, cols in groups.items():
    data = df[cols].dropna(how="all")
    mean = data.mean(axis=1)
    std = data.std(axis=1).fillna(0)
    summary.append(pd.DataFrame({"group": name, "mean": mean, "sd": std, "n": data.notna().sum(axis=1)}))

    plt.plot(mean.index, mean, "-", color=colors[name], linewidth=2.5, label=name)
    plt.plot(mean.index, mean, "o", color=colors[name], markersize=8)
    if len(cols) > 1:
        lower = np.clip(mean - std, a_min=1, a_max=None)
        plt.fill_between(mean.index, lower, mean + std, color=colors[name], alpha=0.2)

plt.yscale("log")
plt.xticks([0, 15, 45, 120, 180, 240], fontsize=16)
plt.yticks(fontsize=16)
plt.xlabel("Time (minutes post infection)", fontsize=17)
plt.ylabel("Count (log scale)", fontsize=17)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=16, frameon=False)
plt.grid(True, which="major", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig1A.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "Fig1A.pdf", bbox_inches="tight")
plt.show()

pd.concat(summary).rename_axis("time_min").to_csv(SRC_DIR / "Fig1A_data.csv")